In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch
from scipy.signal import convolve2d

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [7]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xd, yd = s['xd'], s['yd']

df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers_cor.csv', skipinitialspace=True).dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('ipsf.npz')
ipsf = s['ipsf']


def make_map(files, pole='north', stonyhurst=False, q=0, mu_thr=0.1):
    nx, ny = 1024, 1024
    xc, yc = 511.5, 511.5
    Rsun = 480


    if pole == 'north':
        view_new = View(nx, ny, xc, yc, Rsun, -90, 90, 0, rsun_arc=q * 3600) ### North pole view
    else:
        view_new = View(nx, ny, xc, yc, Rsun, 90, -90, 0, rsun_arc=q * 3600) ### South pole view

    grid = view_new.grid()
    mean, variance, coverage, coverage2 = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

    for file in files[:]:
        print(file)

        with fits.open(file) as hdul:
            header = hdul[0].header.copy()
            data = hdul[0].data.copy()

        #data = convolve2d(data, ipsf, mode='same', boundary='symm')
        #data = undistort(data, header, xd, yd)

        did = int(file.split('_')[-1].split('.')[0])
        xd_, yd_ = crop_grid(xd, yd, header)

        view = View.from_header(header)
        view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.25, rsun=ru_sun[dids == did][0], inplace=True)

        transform = (view_new.to_carrington(correct_mu=True) -
                     view.to_synoptic(correct_mu=True, mu_thr=mu_thr, stonyhurst=stonyhurst))
        grid_, alpha = transform(grid)
        grid_ = (interp2d(xd_, *grid_, kind='bilinear'), interp2d(yd_, *grid_, kind='bilinear'))
        data = interp2d(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 2
        coverage += np.nan_to_num(n)
        coverage2 += np.nan_to_num(n ** 2)
        mean_ = mean.copy()
        mean += np.nan_to_num((data - mean) * n / coverage)
        variance += np.nan_to_num((data - mean) * (data - mean_) * n)


    variance = variance / coverage
    mean[coverage == 0] = np.nan
    coverage[coverage == 0] = np.nan
    coverage2[coverage2 == 0] = np.nan
    variance[coverage == 0] = np.nan

    return mean, variance, coverage, coverage2, view_new

In [8]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()

dates = np.array([datetime.fromisoformat(date) for date in df['date']])
car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()

np.unique(car_rot)

array([2279, 2280, 2281, 2282, 2283, 2284, 2285, 2286, 2287, 2288, 2289,
       2290, 2291, 2292, 2293, 2294, 2295, 2296, 2297, 2298, 2299, 2300,
       2301, 2302, 2303, 2304, 2305, 2306])

In [16]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*')))

t = np.where(car_rot == 2296)[0]

In [17]:
mean, variance, coverage, coverage2, view = make_map(files[t], pole='north', stonyhurst=False)

/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250408T060003_V202602220258_0544080501.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250408T180003_V202602220258_0544080502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250409T060003_V202602220258_0544090501.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250409T180003_V202602220258_0544090502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250410T060002_V202602220258_0544100501.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250410T180003_V202602220258_0544100502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250411T060003_V202602220258_0544110501.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250411T180003_V202602220258_0544110502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250412T060003_V202602220258_0544120501.fits.gz
/home/ulyanov/data/solo/phi/

/tmp/ipykernel_81296/2456222943.py:56: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [18]:
show_data(mean, view, label=r'$B_{los}$', vmin=-100, vmax=100)

In [6]:
variance_ = variance * coverage2 / coverage ** 2
mean_ = mean.copy()
mean_[np.abs(mean) < 3 * np.sqrt(variance_)] = np.nan

/tmp/ipykernel_131505/3052948849.py:3: RuntimeWarning: invalid value encountered in sqrt
  mean_[np.abs(mean) < 3 * np.sqrt(variance_)] = np.nan


In [7]:
show_data(mean_, view, label=r'$B_{los}$', vmin=-100, vmax=100)

In [8]:
show_data(np.sqrt(variance), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=40)

/tmp/ipykernel_131505/2739061088.py:1: RuntimeWarning: invalid value encountered in sqrt
  show_data(np.sqrt(variance), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=40)


In [9]:
show_data(np.sqrt(variance_), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=10)

/tmp/ipykernel_131505/1909495063.py:1: RuntimeWarning: invalid value encountered in sqrt
  show_data(np.sqrt(variance_), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=10)


In [10]:
show_data(np.abs(mean) / np.sqrt(variance_), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=5)

/tmp/ipykernel_131505/3723752687.py:1: RuntimeWarning: invalid value encountered in sqrt
  show_data(np.abs(mean) / np.sqrt(variance_), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=5)


In [11]:
q = 35
sin = np.sin(q * np.pi / 180)

W = 14.712 - 2.396 * sin ** 2 - 1.787 * sin ** 4 - 0.985
360 / W / 27.27 * 360
#W * 31#27.27

np.float64(372.879870302123)